# Week 3 Project 2 — Meeting minutes and action items

This notebook implements the course project as a local pipeline: transcript (or optional local audio) → Ollama structured extraction → validation → Markdown/JSON export. It is intentionally transcript-first so it runs on a CPU without downloading a speech model.

For audio, install `faster-whisper`, set `USE_WHISPER = True`, and point `AUDIO_PATH` at a local file. The rest of the pipeline remains unchanged.


In [ ]:
from pathlib import Path
import json
import re
import requests

OLLAMA_URL = "http://127.0.0.1:11434"
MODEL = "llama3.2:1b"
TIMEOUT_SECONDS = 240
USE_WHISPER = False
AUDIO_PATH = "meeting.mp3"


In [ ]:
def ollama_chat(prompt, system="You are an accurate meeting secretary."):
    response = requests.post(f"{OLLAMA_URL}/api/chat", json={"model": MODEL, "messages": [{"role": "system", "content": system}, {"role": "user", "content": prompt}], "stream": False, "format": "json", "options": {"temperature": 0, "num_ctx": 8192, "num_predict": 1200}}, timeout=TIMEOUT_SECONDS)
    response.raise_for_status()
    return response.json()["message"]["content"].strip()

def parse_json(text):
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.I | re.S).strip()
    try: return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.S)
        if not match: raise ValueError("Ollama returned no JSON object")
        return json.loads(match.group(0))


In [ ]:
def transcribe_audio(audio_path):
    try: from faster_whisper import WhisperModel
    except ImportError as exc: raise RuntimeError("Optional audio path requires: python -m pip install faster-whisper") from exc
    model = WhisperModel("base", device="cpu", compute_type="int8")
    segments, _ = model.transcribe(str(audio_path), vad_filter=True)
    return " ".join(segment.text.strip() for segment in segments).strip()

TRANSCRIPT = """Meeting: Customer support planning, 8 May.
Mona: We need a bilingual FAQ for the refund process before the next release.
Omar: I will draft the English and Arabic examples by 12 May.
Sara: I will review the draft with legal on 15 May.
The team agreed to measure unanswered tickets for two weeks.
Risk: the analytics export may not include archived conversations."""
if USE_WHISPER: transcript = transcribe_audio(AUDIO_PATH)
else: transcript = TRANSCRIPT
print(transcript)


In [ ]:
SCHEMA = {"title": "short meeting title", "summary": ["short factual sentence"], "decisions": ["decision"], "action_items": [{"task": "specific task", "owner": "person or unassigned", "due_date": "date or null"}], "risks": ["risk or open question"]}
def extract_minutes(transcript_text):
    prompt = "Convert the transcript into the schema below. Do not invent owners, dates, decisions, or risks. Use null for an unknown due date. Return only JSON.\n\nSCHEMA:\n" + json.dumps(SCHEMA) + "\n\nTRANSCRIPT:\n" + transcript_text
    data = parse_json(ollama_chat(prompt))
    data.setdefault("title", "Untitled meeting"); data.setdefault("summary", []); data.setdefault("decisions", []); data.setdefault("action_items", []); data.setdefault("risks", [])
    for item in data["action_items"]:
        item.setdefault("task", ""); item.setdefault("owner", "unassigned"); item.setdefault("due_date", None)
    return data
minutes = extract_minutes(transcript)
print(json.dumps(minutes, indent=2, ensure_ascii=False))


In [ ]:
def minutes_markdown(data):
    lines = [f"# {data['title']}", "", "## Summary"]
    lines += [f"- {item}" for item in data["summary"]] or ["- No summary returned."]
    lines += ["", "## Decisions"] + ([f"- {item}" for item in data["decisions"]] or ["- None recorded."])
    lines += ["", "## Action items"]
    lines += ([f"- **{item['owner']}** — {item['task']} (due: {item['due_date'] or 'unspecified'})" for item in data["action_items"]] or ["- None recorded."])
    lines += ["", "## Risks and open questions"] + ([f"- {item}" for item in data["risks"]] or ["- None recorded."])
    return "\n".join(lines) + "\n"
result_dir = Path("results"); result_dir.mkdir(exist_ok=True)
(result_dir / "meeting_minutes.json").write_text(json.dumps(minutes, indent=2, ensure_ascii=False), encoding="utf-8")
(result_dir / "meeting_minutes.md").write_text(minutes_markdown(minutes), encoding="utf-8")
print("Saved results/meeting_minutes.json and results/meeting_minutes.md")


## Optional Gradio UI

Paste a transcript and click **Create minutes**. A remote Jupyter session should be reached through the SSM tunnel documented in `aws/README.md`.


In [ ]:
import gradio as gr
def ui_minutes(text): return minutes_markdown(extract_minutes(text))
RUN_UI = False
if RUN_UI:
    gr.Interface(fn=ui_minutes, inputs=gr.Textbox(value=TRANSCRIPT, lines=12, label="Transcript"), outputs=gr.Markdown(), title="Meeting minutes and action items — local Ollama").launch()
